# 🎨 SF Parks + Public Art - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

Two small, hand-curated datasets about San Francisco's public spaces: every **Recreation and Parks property** (parks, playgrounds, rec centers, community gardens, golf courses) and the city's **Civic Art Collection** (murals, sculptures, and installations in parks, libraries, plazas, and SFO).

**What this notebook does:**
1. Fetches both datasets from DataSF - no API key required
2. Normalises each record and builds a `geo_point`
3. Indexes them into `sf_parks` and `sf_public_art`, then joins them with a geo query

---
> **Data source:** [Recreation and Parks Properties](https://data.sf.gov/d/gtr9-ntp6) (`gtr9-ntp6`) and the San Francisco Arts Commission [Civic Art Collection](https://data.sf.gov/d/r7bn-7v9c) (`r7bn-7v9c`).

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch both datasets
Both are small (a few hundred parks, ~1,000 artworks), so we take everything.

In [ ]:
def fetch_socrata(dataset_id, limit, page_size=10000, **params):
    """Page through a DataSF dataset with $limit/$offset until `limit` rows or the end."""
    url = f'https://data.sf.gov/resource/{dataset_id}.json'
    rows = []
    while len(rows) < limit:
        page = {**params, '$limit': min(page_size, limit - len(rows)), '$offset': len(rows)}
        r = requests.get(url, params=page, timeout=120)
        r.raise_for_status()
        batch = r.json()
        rows.extend(batch)
        print(f'  fetched {len(rows)} rows...', end='\r', flush=True)
        if len(batch) < page['$limit']:
            break
    return rows

def geo_point(lat, lon):
    """Build a geo_point from lat/lon strings, skipping blanks and 0,0."""
    try:
        lat_f, lon_f = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if lat_f == 0.0 or lon_f == 0.0:
        return None
    return {'lat': lat_f, 'lon': lon_f}

parks = fetch_socrata('gtr9-ntp6', 5000)
print(f'\n✅ Fetched {len(parks)} park properties')
art = fetch_socrata('r7bn-7v9c', 5000)
print(f'\n✅ Fetched {len(art)} artworks')

## 3. Index the parks

In [ ]:
INDEX = 'sf_parks'

mapping = {'mappings': {'properties': {
    'property_id':   {'type': 'keyword'},
    'name':          {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'property_type': {'type': 'keyword'},   # Regional Park, Neighborhood Park or Playground, Mini Park, ...
    'acres':         {'type': 'float'},
    'address':       {'type': 'text'},
    'zip':           {'type': 'keyword'},
    'complex':       {'type': 'keyword'},
    'neighborhood':  {'type': 'keyword'},
    'location':      {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

def enrich_park(r):
    try:
        acres = round(float(r.get('acres')), 2)
    except (TypeError, ValueError):
        acres = None
    doc = {
        'property_id':   r.get('property_id'),
        'name':          r.get('property_name'),
        'property_type': r.get('propertytype'),
        'acres':         acres,
        'address':       r.get('address'),
        'zip':           r.get('zipcode'),
        'complex':       r.get('complex'),
        'neighborhood':  r.get('analysis_neighborhood'),
        'location':      geo_point(r.get('latitude'), r.get('longitude')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich_park(r) for r in parks]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 4. Index the public art
Some pieces are in storage or at SFO; `current_location` tells you which. Only pieces on public display in the city have coordinates.

In [ ]:
INDEX = 'sf_public_art'

mapping = {'mappings': {'properties': {
    'accession_number': {'type': 'keyword'},
    'title':            {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'artist':           {'type': 'text', 'fields': {'keyword': {'type': 'keyword'}}},
    'created':          {'type': 'keyword'},   # free text: "2008", "ca. 1939", "1987-1990", ...
    'medium':           {'type': 'text'},
    'media_support':    {'type': 'text'},
    'dimensions':       {'type': 'keyword'},
    'credit_line':      {'type': 'text'},
    'facility':         {'type': 'keyword'},
    'current_location': {'type': 'keyword'},   # Public Display, SFO, ...
    'location_description': {'type': 'text'},
    'address':          {'type': 'text'},
    'cultural_districts': {'type': 'keyword'},
    'neighborhood':     {'type': 'keyword'},
    'location':         {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

def enrich_art(r):
    doc = {
        'accession_number': r.get('accession_number'),
        'title':            r.get('display_title'),
        'artist':           r.get('artist'),
        'created':          r.get('creation_date'),
        'medium':           r.get('medium'),
        'media_support':    r.get('media_support'),
        'dimensions':       r.get('display_dimensions'),
        'credit_line':      r.get('credit_line'),
        'facility':         r.get('facility'),
        'current_location': r.get('current_location'),
        'location_description': r.get('location_description'),
        'address':          r.get('street_address_or_intersection'),
        'cultural_districts': r.get('cultural_districts'),
        'neighborhood':     r.get('analysis_neighborhood'),
        'location':         geo_point(r.get('latitude'), r.get('longitude')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich_art(r) for r in art]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')

## 5. Verify - and join them with a geo query

In [ ]:
resp = es.search(index='sf_parks', body={'size': 0, 'aggs': {'by_type': {'terms': {'field': 'property_type', 'size': 10}}}})
print(f"Parks: {resp['hits']['total']['value']}")
for b in resp['aggregations']['by_type']['buckets']:
    print(f"  {b['key']:40s} {b['doc_count']}")

resp = es.search(index='sf_public_art', body={'size': 0, 'aggs': {
    'by_where': {'terms': {'field': 'current_location', 'size': 5}},
    'by_neighborhood': {'terms': {'field': 'neighborhood', 'size': 5}},
}})
print(f"\nArtworks: {resp['hits']['total']['value']}")
for b in resp['aggregations']['by_where']['buckets']:
    print(f"  {b['key']:40s} {b['doc_count']}")
print('Most art (by neighborhood):')
for b in resp['aggregations']['by_neighborhood']['buckets']:
    print(f"  {b['key']:40s} {b['doc_count']}")

In [ ]:
# Pick a park, then find the public art within 300 m of it
park = es.search(index='sf_parks', body={'size': 1, 'query': {'match': {'name': 'Dolores Park'}}})['hits']['hits'][0]['_source']
resp = es.search(index='sf_public_art', body={'size': 10,
    'query': {'geo_distance': {'distance': '300m', 'location': park['location']}},
    '_source': ['title', 'artist', 'created'],
})
print(f"Public art within 300 m of {park['name']}:")
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"  {s.get('title')} - {s.get('artist')} ({s.get('created', '?')})")

## 6. Going further with Mistral

- **Walking tour:** pick a starting point, use geo queries to find parks and art along the way, and have a Mistral model write the tour narration (and **Voxtral TTS** read it aloud).
- **Semantic art search:** embed title + medium + location description with Mistral embeddings and search by meaning ("mosaic about the ocean", "bronze animal").
- **Accessible descriptions:** generate plain-language or translated descriptions of each piece for visitors.
- **Combine:** add [sf_community_events](sf_community_events.ipynb) to suggest what's happening in the park this weekend, or [sf_landmarks](sf_landmarks.ipynb) for the history nearby.

See [mistral_guide.md](mistral_guide.md) and [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).

---
## ✅ Index loaded
`sf_parks` and `sf_public_art` are ready. Both have a `geo_point`, so you can join them to each other and to every other SF index here by distance.

**To refresh the data**, re-run cells 2-4. Each index is dropped and recreated, so there are no duplicates.